# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using the Croissant schema and is accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the mlcroissant library if it's not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
meta = dataset.metadata
print('Dataset Name:', getattr(meta, 'name', '(unknown)'))
print('Dataset Description:')
print(getattr(meta, 'description', 'No description provided.'))


## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all record set `@id`s and their field (column) `@id`s with descriptions. These `@id`s are what you'll use for extraction and analysis.

In [ ]:
# List available record sets and fields using their @id
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
all_record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'description') and rs.description:
        print(f"  Description: {rs.description}")
    all_record_set_ids.append(rs.id)
    print("  Fields (columns):")
    for f in rs.fields:
        print(f"    - name: {f.name}, @id: {f.id}, dataType: {f.data_type}")
    print()
# For demonstration, pick the first record set
if all_record_set_ids:
    example_record_set_id = all_record_set_ids[0]
    print(f"Example record set for next steps: {example_record_set_id}")
else:
    raise ValueError("No record sets found in this dataset.")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. Always use the record set and field `@id`s from the previous overview.

In [ ]:
# Extract data from all record sets
dataframes = {}
for rs_id in all_record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"Loaded {df.shape[0]} records for RecordSet (@id={rs_id}). Columns:")
    print(df.columns.tolist())
    print()
# Select our main record set for exploration
main_record_set_id = example_record_set_id
print(f"Using {main_record_set_id} for EDA.")
# Show a sample
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field using its `@id`, filter the data, normalize the selected numeric field, and optionally group by a categorical field (also referenced by `@id`).

In [ ]:
# Select a numeric field for analysis using its @id
df = dataframes[main_record_set_id]
record_set_obj = None
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        record_set_obj = rs
        break
if not record_set_obj:
    raise ValueError(f"RecordSet with @id={main_record_set_id} not found.")

# Find a suitable numeric field (preferably 'MW' (molecular weight), 'Abundance', or similar)
numeric_field = None
for f in record_set_obj.fields:
    if f.data_type in ('Float', 'Number', 'Integer'):
        numeric_field = f.id
        break
if not numeric_field:
    raise ValueError("No numeric field found in record set.")

print(f"Chosen numeric field for analysis: {numeric_field}")

# Filtering
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold].copy()
else:
    raise ValueError(f"Field {numeric_field} not found in DataFrame columns.")

print(f"Filtered {filtered_df.shape[0]} records from {df.shape[0]} where {numeric_field} > {threshold}.")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nFirst 5 normalized values for {numeric_field}:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (string, not same as numeric_field, and not @id)
group_field = None
for f in record_set_obj.fields:
    if f.data_type.lower() in ("string", "text") and f.id != numeric_field:
        group_field = f.id
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = (
        filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    )
    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo suitable text group field found or field not present for grouping.")

## 5. Visualization

We'll visualize the distribution of the numeric field and (if grouped above) compare means across categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (> {threshold})")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouping done, plot group means
if 'grouped_df' in locals() and group_field and group_field in grouped_df.columns:
    plt.figure(figsize=(8,5))
    top_n = grouped_df.head(15)
    sns.barplot(data=top_n, x=numeric_field, y=group_field)
    plt.title(f"Mean {numeric_field} by {group_field} (top 15)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the Croissant dataset standard and the `mlcroissant` library to:
- Load the dataset and metadata
- Explore the record sets and the schema via `@id`
- Extract data to DataFrames
- Analyze and process numeric fields
- Visualize key data distributions

This workflow can be extended to deeper downstream scientific, machine learning, or domain-specific analyses tailored to the dataset's protein abundance and characterization features.